In [ ]:
import glob, os, shutil, subprocess, tempfile
from pathlib import Path
OCTAVE = shutil.which("octave-cli") or shutil.which("octave")
if OCTAVE is None and os.name == "nt":
    candidates = glob.glob(r"C:\Program Files\GNU Octave\Octave-*\mingw64\bin\octave-cli.exe")
    if candidates:
        OCTAVE = sorted(candidates)[-1]
if OCTAVE is None:
    raise RuntimeError("Octave CLI not found. Add octave-cli to PATH or install GNU Octave.")
WORK = Path(tempfile.mkdtemp(prefix="me256-coursework-"))
print(f"Octave: {OCTAVE}")
print(f"Temporary source directory: {WORK}")

for relative in ['vmso3', 'vmso31']:
    (WORK / relative).mkdir(parents=True, exist_ok=True)


# ME256 Homework 3: variational models for flexible structures

**Official statement:** [ME256 Homework 3 (2011)](https://mecheng.iisc.ac.in/suresh/me256/hw2011/homework3.html). The outline below paraphrases the tasks for study; consult the linked page for the complete statement and exact requirements.

The assignment has four modeling questions:

1. Formulate a variational model for a flexible wire of fixed length $L$, initially on the horizontal axis with one endpoint at the origin and the other at $(L,0)$. The free endpoint is moved vertically to $(L,L)$. Identify a suitable function space and norm, check the norm axioms, and discuss what can be said about the solution.
2. Extend the wire model to a flexible cloth that bends but does not stretch; outline the formulation, with a more complete derivation as an optional extension.
3. Construct a sufficiently motivated variational example that uses an inner product, not only a norm.
4. Formulate a structural-optimization problem in a variational framework and explain the function space and norm or inner product needed for its functionals.

The archived MATLAB code is relevant chiefly to the first two topics: it discretizes the wire and cloth configurations, measures deviation from a reference state, and uses equality constraints to approximate fixed segment or mesh-edge lengths. The last two questions are open-ended modeling work and are not answered by these code listings alone.

## Study prompts

- Identify the continuous configuration and admissible function space suggested by the discrete variables, and verify the norm properties if a norm is proposed.
- Explain how the length-preservation constraints approximate inextensibility, then consider what changes as the cloth mesh is refined.
- Develop separate examples for an inner-product-based functional and a structural-optimization functional.
- Relate numerical objectives and constraints back to the modeling assumptions in the official statement.

## Reproducibility and version history

This notebook uses a Python 3 Jupyter kernel to materialize the embedded MATLAB source into a temporary directory, then invokes Octave CLI. The `.m` sources are embedded below; they are not stored separately in this repository. Install Jupyter/IPython and an Octave installation with the `optim` package. MATLAB with Optimization Toolbox remains the closest match to the original runtime. The variant code PDFs have matching filenames but different contents, so both versions are embedded and tested separately. In `vmso31`, one edge direction is absent; that variant is preserved rather than silently repaired.

## Variant `vmso3`

This version uses the coarser cloth discretization. Its constraint routine includes horizontal and vertical edges and both diagonals in each cell.

## Variant `vmso31`

This later-looking folder has a finer cloth discretization but a different set of distance constraints. Preserve it as a separate historical variant; do not assume the names alone indicate which copy is authoritative.

### Separate improvement ideas from historical code

A useful follow-up study would compare mesh-refinement behavior and test whether the intended cloth constraints are complete. Keep such experiments in new files or a new variant so the reconstructed originals remain traceable.

## Embedded source code

Each following code cell contains one MATLAB source file and writes it into a temporary working directory. Run the notebook from top to bottom; no reconstructed `.m` files are required in the repository.

### `vmso3/wire2d_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'wire2d_simulation.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize a segmented wire configuration and plot its final shape.
clc;close all;clear all;
global n d zeta head_movement
L=1;
n=40;
d=L/n;
head_movement=L;
zeta=zeros(1,n+1);
for i=1:1:n+1
    zeta(i)=(i-1)*L/n;
end
theta0=zeros(n+1,1);
[theta,f,exitflag,options]=fmincon(@wire2d,theta0,[],[],[],[],zeros(n+1,1),ones(n+1,1)*2*pi,[],optimset('Algorithm','active-set'));
fprintf('Wire objective %.6g; optimizer exit flag %d\n', f, exitflag);
x=zeros(1,n+1);
y=zeros(1,n+1);
x(n+1)=zeta(n+1);
y(n+1)=head_movement;
for j=n:-1:1
    x(j)=x(j+1)-d*cos(theta(j));
    y(j)=y(j+1)-d*sin(theta(j));
end
plot(x,y)
hold on
plot(zeta,zeros(1,n+1),'r')
axis image
legend('Final Configuration','Initial Configuration','Location','NorthEastOutside')
title('Wire Simulation')


### `vmso3/wire2d.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'wire2d.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize the wire's total squared displacement from its start.
function f=wire2d(theta)
    global n d zeta head_movement
    x=zeros(1,n+1);
    y=zeros(1,n+1);
    x(n+1)=zeta(n+1);
    y(n+1)=head_movement;
    for j=n:-1:1
        x(j)=x(j+1)-d*cos(theta(j));
        y(j)=y(j+1)-d*sin(theta(j));
    end
    f=0;
    for k=n+1:-1:1
        f=f+((x(k)-zeta(k))^2+y(k)^2);
    end
    f=sqrt(f);


### `vmso3/cloth_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'cloth_simulation.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize and plot a triangulated cloth mesh with fixed edge lengths.
clc;close all;clear all;
global n d zeta eta corner_movement k l
L=1;
n=3;
d=L/n;
corner_movement=L;
zeta=zeros(1,n+1);
eta=zeros(1,n+1);
k=zeros(1,(n+1)*(n+1));
l=zeros(1,(n+1)*(n+1));
for i=1:1:n+1
    for j=1:1:n+1
        k((i-1)*(n+1)+j)=j;
        l((i-1)*(n+1)+j)=i;
    end
end
for i=1:1:n+1
    zeta(i)=(i-1)*L/n;
    eta(i)=(i-1)*L/n;
end
Aeq=zeros(3,3*(n+1)^2);
Aeq(1,(n+1)^2)=1;
Aeq(2,2*(n+1)^2)=1;
Aeq(3,3*(n+1)^2)=1;
beq=[zeta(n+1);eta(n+1);corner_movement];
r0=[(k-1)*L/n, (l-1)*L/n, ones(1,(n+1)*(n+1))*corner_movement];
[r,fval,exitflag,output]=fmincon(@cloth,r0,[],[],Aeq,beq,[],[],@dist,optimset('Algorithm','interior-point'));
fprintf('Cloth objective %.6g; optimizer exit flag %d\n', fval, exitflag);
x=r(1:(n+1)^2);
y=r((n+1)^2+1:2*(n+1)^2);
z=r(2*(n+1)^2+1:3*(n+1)^2);
for i=1:1:n
    for j=1:1:n
        patch([x(j+(i-1)*(n+1));x(j+1+(i-1)*(n+1));x(j+i*(n+1))],...
        [y(j+(i-1)*(n+1));y(j+1+(i-1)*(n+1));y(j+i*(n+1))],...
        [z(j+(i-1)*(n+1));z(j+1+(i-1)*(n+1));z(j+i* (n+1))],'b','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.4)
        end
        patch([x(j+1+(i-1)*(n+1));x(j+i*(n+1));x(j+1+i*(n+1))],...
        [y(j+1+(i-1)*(n+1));y(j+i*(n+1));y(j+1+i*(n+1))],...
        [z(j+1+(i-1)*(n+1));z(j+i*(n+1));z(j+1+i* (n+1))],'r','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.4)
        end

    end
end
grid on


### `vmso3/cloth.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'cloth.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize mesh-node displacement from the initial cloth plane.
function f=cloth(r)
    global n zeta eta k l
    f=0;
    for i=1:1:(n+1)*(n+1)
        f=f+((r(i)-zeta(k(i)))^2+(r(i+(n+1)*(n+1))-eta(l(i)))^2+(r(i+2*(n+1)*(n+1))) ^2);
    end
    f=sqrt(f);


### `vmso3/dist.m`

In [ ]:
%%writefile {WORK / 'vmso3' / 'dist.m'}
% Reconstructed from the vmso3 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Equality constraints preserve horizontal, vertical, and diagonal mesh lengths.
function [c,ceq]=dist(r)
    global n d
    ceq=zeros(2*n*(n+1)+2*n^2,1);
    k=1;
    for j=1:1:n+1
        for i=1:1:n
            ceq(k)=sqrt((r(i+1+(n+1)*(j-1))-r(i+(n+1)*(j-1)))^2+...
            (r(i+1+(n+1)*(j-1)+(n+1)*(n+1))-r(i+(n+1)*(j-1)+(n+1)*(n+1)))^2+...
            (r(i+1+(n+1)*(j-1)+2*(n+1)*(n+1))-r(i+(n+1)*(j-1)+2*(n+1)*(n+1)))^2)- d;
            k=k+1;
        end
    end
    for i=1:1:n+1
        for j=1:1:n
            ceq(k)=sqrt((r(i+(n+1)*j)-r(i+(n+1)*(j-1)))^2+...
            (r(i+(n+1)*j+(n+1)*(n+1))-r(i+(n+1)*(j-1)+(n+1)*(n+1)))^2+...
            (r(i+(n+1)*j+2*(n+1)*(n+1))-r(i+(n+1)*(j-1)+2*(n+1)*(n+1)))^2)-d;
            k=k+1;
        end
    end
    for j=1:1:n
        for i=1:1:n
            ceq(k)=sqrt((r(i+1+(n+1)*j)-r(i+(n+1)*(j-1)))^2+...
            (r(i+1+(n+1)*j+(n+1)^2)-r(i+(n+1)*(j-1)+(n+1)^2))^2+...
            (r(i+1+(n+1)*j+2*(n+1)^2)-r(i+(n+1)*(j-1)+2*(n+1)^2))^2)-sqrt(2)*d;
            ceq(k+1)=sqrt((r(i+(n+1)*j)-r(i+1+(n+1)*(j-1)))^2+...
            (r(i+(n+1)*j+(n+1)^2)-r(i+1+(n+1)*(j-1)+(n+1)^2))^2+...
            (r(i+(n+1)*j+2*(n+1)^2)-r(i+1+(n+1)*(j-1)+2*(n+1)^2))^2)-sqrt(2)*d;
            k=k+1;
        end
    end
    c=[];


### `vmso31/wire2d_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'wire2d_simulation.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize a segmented wire configuration and plot its final shape.
clc;close all;clear all;
global n d zeta head_movement
L=1;
n=40;
d=L/n;
head_movement=L;
zeta=zeros(1,n+1);
for i=1:1:n+1
    zeta(i)=(i-1)*L/n;
end
theta0=zeros(n+1,1);
[theta,f,exitflag,options]=fmincon(@wire2d,theta0,[],[],[],[],zeros(n+1,1),ones(n+1,1)*2*pi,[],optimset('Algorithm','active-set'));
fprintf('Wire objective %.6g; optimizer exit flag %d\n', f, exitflag);
x=zeros(1,n+1);
y=zeros(1,n+1);
x(n+1)=zeta(n+1);
y(n+1)=head_movement;
for j=n:-1:1
    x(j)=x(j+1)-d*cos(theta(j));
    y(j)=y(j+1)-d*sin(theta(j));
end
plot(x,y)
hold on
plot(zeta,zeros(1,n+1),'r')
axis image
legend('Final Configuration','Initial Configuration','Location','NorthEastOutside')
title('Wire Simulation')


### `vmso31/wire2d.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'wire2d.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize the wire's total squared displacement from its start.
function f=wire2d(theta)
    global n d zeta head_movement
    x=zeros(1,n+1);
    y=zeros(1,n+1);
    x(n+1)=zeta(n+1);
    y(n+1)=head_movement;
    for j=n:-1:1
        x(j)=x(j+1)-d*cos(theta(j));
        y(j)=y(j+1)-d*sin(theta(j));
    end
    f=0;
    for k=n+1:-1:1
        f=f+((x(k)-zeta(k))^2+y(k)^2);
    end
    f=sqrt(f);


### `vmso31/cloth_simulation.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'cloth_simulation.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Optimize and plot a triangulated cloth mesh with distance constraints.
clc;close all;clear all;
global n d zeta eta corner_movement k l
L=1;
n=10;
d=L/n;
corner_movement=L;
zeta=zeros(1,n+1);
eta=zeros(1,n+1);
k=zeros(1,(n+1)*(n+1));
l=zeros(1,(n+1)*(n+1));
for i=1:1:n+1
    for j=1:1:n+1
        k((i-1)*(n+1)+j)=j;
        l((i-1)*(n+1)+j)=i;
    end
end
for i=1:1:n+1
    zeta(i)=(i-1)*L/n;
    eta(i)=(i-1)*L/n;
end
Aeq=zeros(3,3*(n+1)^2);
Aeq(1,(n+1)^2)=1;
Aeq(2,2*(n+1)^2)=1;
Aeq(3,3*(n+1)^2)=1;
beq=[zeta(n+1);eta(n+1);corner_movement];
r0=[(k-1)*L/n, (l-1)*L/n, ones(1,(n+1)*(n+1))*corner_movement];
[r,fval,exitflag,output]=fmincon(@cloth,r0,[],[],Aeq,beq,[],[],@dist,optimset('Algorithm','interior-point'));
fprintf('Cloth objective %.6g; optimizer exit flag %d\n', fval, exitflag);
x=r(1:(n+1)^2);
y=r((n+1)^2+1:2*(n+1)^2);
z=r(2*(n+1)^2+1:3*(n+1)^2);
for i=1:1:n
    for j=1:1:n
        patch([x(j+(i-1)*(n+1));x(j+1+(i-1)*(n+1));x(j+i*(n+1))],...
        [y(j+(i-1)*(n+1));y(j+1+(i-1)*(n+1));y(j+i*(n+1))],...
        [z(j+(i-1)*(n+1));z(j+1+(i-1)*(n+1));z(j+i* (n+1))],'b','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.5)
        end
        patch([x(j+1+(i-1)*(n+1));x(j+i*(n+1));x(j+1+i*(n+1))],...
        [y(j+1+(i-1)*(n+1));y(j+i*(n+1));y(j+1+i*(n+1))],...
        [z(j+1+(i-1)*(n+1));z(j+i*(n+1));z(j+1+i* (n+1))],'r','EdgeColor','none')
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.5)
        end

    end
end
grid on


### `vmso31/cloth.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'cloth.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Objective: minimize mesh-node displacement from the initial cloth plane.
function f=cloth(r)
    global n zeta eta k l
    f=0;
    for i=1:1:(n+1)*(n+1)
        f=f+((r(i)-zeta(k(i)))^2+(r(i+(n+1)*(n+1))-eta(l(i)))^2+(r(i+2*(n+1)*(n+1))) ^2);
    end
    f=sqrt(f);


### `vmso31/dist.m`

In [ ]:
%%writefile {WORK / 'vmso31' / 'dist.m'}
% Reconstructed from the vmso31 code3.pdf.
% Problem-solving approach is retained; see the assignment README.
% Original variant constraints: one edge direction and both cell diagonals.
function [c,ceq]=dist(r)
    global n d
    ceq=zeros(n*(n+1)+2*n^2,1);
    k=1;
    for j=1:1:n+1
        for i=1:1:n
            ceq(k)=sqrt((r(i+1+(n+1)*(j-1))-r(i+(n+1)*(j-1)))^2+...
            (r(i+1+(n+1)*(j-1)+(n+1)*(n+1))-r(i+(n+1)*(j-1)+(n+1)*(n+1)))^2+...
            (r(i+1+(n+1)*(j-1)+2*(n+1)*(n+1))-r(i+(n+1)*(j-1)+2*(n+1)*(n+1)))^2)- d;
            k=k+1;
        end
    end
    for j=1:1:n
        for i=1:1:n
            ceq(k)=sqrt((r(i+1+(n+1)*j)-r(i+(n+1)*(j-1)))^2+...
            (r(i+1+(n+1)*j+(n+1)^2)-r(i+(n+1)*(j-1)+(n+1)^2))^2+...
            (r(i+1+(n+1)*j+2*(n+1)^2)-r(i+(n+1)*(j-1)+2*(n+1)^2))^2)-sqrt(2)*d;
            ceq(k+1)=sqrt((r(i+(n+1)*j)-r(i+1+(n+1)*(j-1)))^2+...
            (r(i+(n+1)*j+(n+1)^2)-r(i+1+(n+1)*(j-1)+(n+1)^2))^2+...
            (r(i+(n+1)*j+2*(n+1)^2)-r(i+1+(n+1)*(j-1)+2*(n+1)^2))^2)-sqrt(2)*d;
            k=k+1;
        end
    end
    c=[];


## Octave 11.3.0 test results

Both wire scripts execute, but Octave `optim` returns exit flag 0 at its 100-iteration limit, so their runs did not converge. Both cloth scripts fail in Octave `fmincon` with a singular-matrix warning followed by “could not regain binding constraints.” This is recorded rather than hidden or worked around by changing the optimizer or constraint model. The printed input configuration satisfies the prescribed length equalities; the optimizer failure needs comparison with MATLAB and review of the original formulation. The two variants remain distinct.

Run statuses and stderr are captured in the cell output below.

In [ ]:
import subprocess

def run_case(label, folder, script, needs_optim=True):
    setup = "set(0,'defaultfigurevisible','off');"
    if needs_optim:
        setup += "pkg load optim;"
    setup += "if any(strcmp(available_graphics_toolkits(),'gnuplot')), graphics_toolkit('gnuplot'); end;"
    setup += f"run('{script}');"
    result = subprocess.run(
        [OCTAVE, "--no-gui", "--quiet", "--eval", setup],
        cwd=WORK / folder, text=True, capture_output=True, errors="replace"
    )
    status = "PASS" if result.returncode == 0 else "FAILED"
    print(f"\n=== {label}: {status} (exit code {result.returncode}) ===")
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    return result.returncode

run_case('vmso3 wire', 'vmso3', 'wire2d_simulation.m', True)
run_case('vmso3 cloth', 'vmso3', 'cloth_simulation.m', True)
run_case('vmso31 wire', 'vmso31', 'wire2d_simulation.m', True)
run_case('vmso31 cloth', 'vmso31', 'cloth_simulation.m', True)



=== vmso3 wire: PASS (exit code 0) ===
Wire objective 4.34721; optimizer exit flag 0
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communication with gnuplot
uses a one-directional pipe and limited information is passed back to the
Octave interpreter so most changes made interactively in the plot window
will not be reflected in the graphics properties managed by Octave.  For
example, if the plot window is closed with a mouse click, Octave will not
be notified and will not update its internal list of open figure windows.
The qt toolkit is recommended instead.

The FLTK graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.
The qt toolkit is recommended instead.

=== vmso3 cloth: FAILED (exit code 1) ===
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communica

In [ ]:
import shutil
shutil.rmtree(WORK)
print("Temporary generated source files removed.")